In [ ]:
!pip install segment-anything opencv-python matplotlib
!pip install git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamPredictor
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
predictor = SamPredictor(sam)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
image = cv2.imread("outfit.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

h, w, _ = image.shape

In [ ]:
predictor.set_image(image)

In [ ]:
# Mode: foreground (1) or background (0)
mode_toggle = widgets.ToggleButtons(
    options=[('Foreground (keep)', 1), ('Background (remove)', 0)],
    value=1,
    description='Mode:'
)

add_point_btn = widgets.Button(description="Add Point")
reset_btn = widgets.Button(description="Reset")
save_btn = widgets.Button(description="Save Result")

display(mode_toggle, add_point_btn, reset_btn, save_btn)

In [ ]:
points = []
labels = []

In [ ]:
def add_point(b):
    x = x_slider.value
    y = y_slider.value

    points.append([x, y])
    labels.append(mode_toggle.value)

    update_segmentation()

add_point_btn.on_click(add_point)

In [ ]:
def reset_all(b):
    global points, labels
    points = []
    labels = []
    update_segmentation()

reset_btn.on_click(reset_all)

In [ ]:
def update_segmentation(change=None):
    clear_output(wait=True)
    display(x_slider, y_slider, mode_toggle, add_point_btn, reset_btn, save_btn)

    if len(points) == 0:
        plt.imshow(image)
        plt.title("No points yet")
        plt.axis("off")
        plt.show()
        return

    input_point = np.array(points)
    input_label = np.array(labels)

    masks, scores, _ = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True
    )

    best_mask = masks[np.argmax(scores)]

    plt.imshow(image)
    plt.imshow(best_mask, alpha=0.5)

    # draw all points
    for (px, py), lab in zip(points, labels):
        color = 'green' if lab == 1 else 'red'
        plt.scatter(px, py, color=color, s=100)

    plt.title("Advanced Segmentation")
    plt.axis("off")
    plt.show()

In [ ]:
def save_result(b):
    if len(points) == 0:
        print("No segmentation to save")
        return

    input_point = np.array(points)
    input_label = np.array(labels)

    masks, scores, _ = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True
    )

    best_mask = masks[np.argmax(scores)]

    segmented = image * best_mask[:, :, None]

    cv2.imwrite("mask.png", best_mask.astype(np.uint8)*255)
    cv2.imwrite("segmented.png", cv2.cvtColor(segmented, cv2.COLOR_RGB2BGR))

    print("Saved: mask.png and segmented.png")

save_btn.on_click(save_result)

In [ ]:
from IPython.display import Image, display

display(Image("mask.png"))


display(Image("segmented.png"))